In [2]:
import json
from pathlib import Path

from rag_agent_system.config import load_config
from rag_agent_system.retrieval import FaissRetriever


PROJECT_ROOT = Path(r"C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system")
benchmark_path = (
    PROJECT_ROOT / "data" / "benchmark" / "tinystories_queries.jsonl"
)
targets = {
    "q007": "1500",
    "q038": "9250",
}


def main() -> None:
    cases = {}

    with benchmark_path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            if item["query_id"] in targets:
                cases[item["query_id"]] = item

    missing = set(targets) - set(cases)
    if missing:
        raise ValueError(f"缺少问题：{sorted(missing)}")

    config = load_config(PROJECT_ROOT / "configs" / "default.yaml")

    with config.paths.chunks.open("r", encoding="utf-8-sig") as f:
        chunks = json.load(f)

    chunk_count = len(chunks)
    if chunk_count == 0:
        raise ValueError("语料为空")

    retriever = FaissRetriever(
        index_path=config.paths.faiss_index,
        chunks_path=config.paths.chunks,
        embedding_model=config.retrieval.embedding_model,
        normalize_embeddings=config.retrieval.normalize_embeddings,
    )

    for query_id, target_chunk_id in targets.items():
        question = cases[query_id]["query"]
        results = retriever.retrieve(question, top_k=chunk_count)

        print(f"\nQuestion ID: {query_id}")
        print(f"Question: {question}")
        print(f"Returned chunks: {len(results)}")

        if not results:
            print("没有检索结果")
            continue

        print(f"Top1 score: {results[0].score:.6f}")

        if len(results) >= 50:
            print(f"Top50 score: {results[49].score:.6f}")

        for position, item in enumerate(results, start=1):
            if item.chunk.chunk_id == target_chunk_id:
                print(f"Target chunk: {target_chunk_id}")
                print(f"Target rank: {position}")
                print(f"Target score: {item.score:.6f}")
                print(f"Inside Top50: {position <= 50}")
                break
        else:
            print(f"未找到目标 chunk：{target_chunk_id}")


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Question ID: q007
Question: Who slid down together and hugged after Max ran away from the slide?
Returned chunks: 10417
Top1 score: 0.578681
Top50 score: 0.463859
Target chunk: 1500
Target rank: 308
Target score: 0.385047
Inside Top50: False

Question ID: q038
Question: Who examined the bird's wing and gave it medicine after the cat took it directly to another animal for help?
Returned chunks: 10417
Top1 score: 0.670667
Top50 score: 0.499812
Target chunk: 9250
Target rank: 59
Target score: 0.494427
Inside Top50: False
